# Silver MERGE with a durable ingestion watermark
Run Bronze first. The cursor is Bronze `_load_ts`, NOT business `updated_at`, so late-arriving business updates are considered. Boundary rows are replayed; the business-time guard rejects stale updates. Run a single writer per pipeline. Delta does not provide a multi-table transaction: merge first, watermark second, and replay after a crash. Retention/deletions and equal-business-time conflicting events require a source-specific CDC policy in production.


In [ ]:
bronze_table = 'dbo.bronze_permits'
silver_table = 'dbo.silver_permits'
watermark_table = 'dbo.etl_watermarks'
pipeline_key = 'permits_history_to_silver'


In [ ]:
import re
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

for table in (bronze_table, silver_table, watermark_table):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*', table):
        raise ValueError('Use schema.table identifiers only.')
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.sql(f'CREATE TABLE IF NOT EXISTS {watermark_table} (pipeline STRING, watermark TIMESTAMP) USING DELTA')
cursors = spark.table(watermark_table).filter(F.col('pipeline') == pipeline_key).collect()
if len(cursors) > 1:
    raise ValueError('Duplicate pipeline watermarks; stop concurrent writers and repair the cursor.')
last = cursors[0]['watermark'] if cursors else datetime(1970, 1, 1)
batch = spark.table(bronze_table).filter(F.col('_load_ts') >= F.lit(last)).cache()
try:
    if not batch.isEmpty():
        if batch.filter(F.col('permit_id').isNull() | (F.trim('permit_id') == '') | F.col('updated_at').isNull()).limit(1).count():
            raise ValueError('Invalid business key or event time; watermark not advanced.')
        upper = batch.agg(F.max('_load_ts')).first()[0]
        order = Window.partitionBy('permit_id').orderBy(F.col('updated_at').desc(), F.col('_load_ts').desc(), F.col('_source_file').desc(), F.col('status').desc())
        latest = (batch.withColumn('_rn', F.row_number().over(order)).filter('_rn = 1').drop('_rn')
            .withColumn('status', F.upper(F.trim('status'))))
        if not spark.catalog.tableExists(silver_table):
            latest.limit(0).write.format('delta').saveAsTable(silver_table)
        # Event-time protection prevents late, stale history from overwriting newer state.
        (DeltaTable.forName(spark, silver_table).alias('t').merge(latest.alias('s'), 't.permit_id = s.permit_id')
            .whenMatchedUpdateAll(condition='s.updated_at > t.updated_at')
            .whenNotMatchedInsertAll().execute())
        cursor = spark.createDataFrame([(pipeline_key, upper)], 'pipeline string, watermark timestamp')
        (DeltaTable.forName(spark, watermark_table).alias('t').merge(cursor.alias('s'), 't.pipeline = s.pipeline')
            .whenMatchedUpdateAll(condition='s.watermark >= t.watermark')
            .whenNotMatchedInsertAll().execute())
finally:
    batch.unpersist()
display(spark.table(silver_table))
